# CrossHateTN — socle de données et baseline E1

Ce notebook lance le dépôt `crosshatetn` de bout en bout :

1. récupère le code (dépôt GitHub, ou zip du dépôt ajouté comme dataset Kaggle) ;
2. lance les tests ;
3. construit le socle de données : sources figées et vérifiées, doublons, conflits, quasi-doublons, **splits gelés**, audit des fuites ;
4. évalue les baselines **sur la validation uniquement** ;
5. garde le test verrouillé jusqu'à la fin du projet.

**Avant de lancer** : active Internet dans les options de session (panneau de droite). Pas besoin de GPU. Durée : environ 3 minutes.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/TON-COMPTE/crosshatetn.git"  # à remplacer après avoir poussé le dépôt

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = WORK / "crosshatetn"

if not REPO.exists():
    # 1) Zip du dépôt ajouté comme dataset (Add Input) : Kaggle le décompresse dans /kaggle/input
    uploaded = []
    if Path("/kaggle/input").exists():
        uploaded = [p.parent for p in Path("/kaggle/input").rglob("pyproject.toml")
                    if (p.parent / "src" / "crosshatetn").is_dir()]
    if uploaded:
        shutil.copytree(uploaded[0], REPO)
    # 2) Sinon, clonage depuis GitHub
    elif "TON-COMPTE" not in REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    else:
        raise RuntimeError("Renseigne REPO_URL ou ajoute le zip du dépôt via Add Input, puis relance.")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dépôt prêt :", REPO)

## 1. Tests

Ils vérifient la normalisation, la lecture des sources, les doublons et conflits, le gel des splits, l'audit des fuites et les métriques. Si un test échoue, on s'arrête là.

In [ ]:
tests = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(tests.stdout[-1500:], tests.stderr[-1500:])
assert tests.returncode == 0, "Des tests échouent : ne pas continuer."

## 2. Socle de données

Les sources sont téléchargées à un commit figé et vérifiées par SHA-256. Les splits viennent de `splits/affectation.csv` (versionné) : ils sont identiques pour toute l'équipe. L'audit arrête tout si une fuite est détectée entre splits.

In [ ]:
import pandas as pd
from IPython.display import display

from crosshatetn.pipeline import prepare

prep = prepare("configs/data.yaml")
display(prep.summary())
display(prep.split_table())
display(prep.audit.table())

Les conflits d'étiquettes sont exclus tant qu'ils ne sont pas arbitrés. Pour arbitrer : remplir `annotations/arbitrage.csv` (une ligne par `id_representant`), relancer, puis versionner `annotations/` et `splits/`.

In [ ]:
display(prep.conflicts[["id_representant", "source", "etiquettes", "majorite_stricte", "texte"]])
print(f"{len(prep.audit.review_pairs)} paires proches (similarité dans [0,60 ; 0,70[) listées pour revue, non bloquantes.")
print(f"Réserve Arabizi : {len(prep.pool)} phrases TUNIZI à annoter plus tard.")

## 3. Baselines E1 (validation tunisienne uniquement)

Majoritaire, TF-IDF + régression logistique, TF-IDF + SVM linéaire ; trois variantes de texte (brut, léger, fort) ; deux jeux d'entraînement (tunisien seul, tunisien + levantin). Intervalles de confiance et comparaisons par bootstrap apparié.

In [ ]:
from crosshatetn import baseline

res = baseline.run(prep.labeled, prep.cfg, prep.cfg.path("outputs") / "baseline")
display(res.best.round(3))
display(res.comparisons.round(3))
print("Référence E1 :", res.reference)
print(res.report)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(res.confusion.to_numpy(), display_labels=res.confusion.index).plot(cmap="Blues")
plt.title("Référence E1 (SVM, tunisien seul) — validation")
plt.show()
print(f"{len(res.errors)} erreurs exportées dans outputs/baseline/erreurs_validation_a_categoriser.csv")

## 4. Test gelé

À lancer **une seule fois**, quand tous les choix sont figés. Chaque évaluation est inscrite dans `splits/journal_test.csv`, à versionner : toute l'équipe voit que le test a été utilisé.

In [ ]:
EVALUER_TEST = False  # passer à True une seule fois, en fin de projet

if EVALUER_TEST:
    display(pd.Series(baseline.final_test(prep.labeled, prep.cfg, res.reference, confirm=True)))
else:
    print("Test gelé : non évalué.")

archive = shutil.make_archive(str(WORK / "crosshatetn_sorties"), "zip", REPO / "outputs")
print("Sorties à télécharger :", archive)